In [0]:
# ADLS access permission
storage_account = "storage_account_name"
access_key = "storage_account_key"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

In [0]:
# Read Processed Data
presentation_df = spark.read \
.format("delta") \
.load(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/drivers"
)

display(presentation_df)

broadcast_name,country_code,driver_number,first_name,full_name,headshot_url,last_name,meeting_key,name_acronym,session_key,team_colour,team_name
L NORRIS,,1,Lando,Lando NORRIS,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANNOR01_Lando_Norris/lannor01.png.transform/1col/image.png,Norris,1287,NOR,11307,F47600,McLaren
M VERSTAPPEN,,3,Max,Max VERSTAPPEN,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/M/MAXVER01_Max_Verstappen/maxver01.png.transform/1col/image.png,Verstappen,1287,VER,11307,4781D7,Red Bull Racing
G BORTOLETO,,5,Gabriel,Gabriel BORTOLETO,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/G/GABBOR01_Gabriel_Bortoleto/gabbor01.png.transform/1col/image.png,Bortoleto,1287,BOR,11307,F50537,Audi
I HADJAR,,6,Isack,Isack HADJAR,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/I/ISAHAD01_Isack_Hadjar/isahad01.png.transform/1col/image.png,Hadjar,1287,HAD,11307,4781D7,Red Bull Racing
P GASLY,,10,Pierre,Pierre GASLY,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/P/PIEGAS01_Pierre_Gasly/piegas01.png.transform/1col/image.png,Gasly,1287,GAS,11307,00A1E8,Alpine
S PEREZ,,11,Sergio,Sergio PEREZ,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/S/SERPER01_Sergio_Perez/serper01.png.transform/1col/image.png,Perez,1287,PER,11307,909090,Cadillac
K ANTONELLI,,12,Kimi,Kimi ANTONELLI,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/K/ANDANT01_Kimi_Antonelli/andant01.png.transform/1col/image.png,Antonelli,1287,ANT,11307,00D7B6,Mercedes
F ALONSO,,14,Fernando,Fernando ALONSO,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/F/FERALO01_Fernando_Alonso/feralo01.png.transform/1col/image.png,Alonso,1287,ALO,11307,229971,Aston Martin
C LECLERC,,16,Charles,Charles LECLERC,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/C/CHALEC01_Charles_Leclerc/chalec01.png.transform/1col/image.png,Leclerc,1287,LEC,11307,ED1131,Ferrari
L STROLL,,18,Lance,Lance STROLL,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANSTR01_Lance_Stroll/lanstr01.png.transform/1col/image.png,Stroll,1287,STR,11307,229971,Aston Martin


In [0]:
# Selecting Required Columns
from pyspark.sql.functions import col

presentation_df = presentation_df.select(
    col("driver_number"),
    col("full_name"),
    col("team_name"),
    col("country_code"),
    col("team_colour"),
    col("name_acronym")
)

display(presentation_df)

driver_number,full_name,team_name,country_code,team_colour,name_acronym
1,Lando NORRIS,McLaren,,F47600,NOR
3,Max VERSTAPPEN,Red Bull Racing,,4781D7,VER
5,Gabriel BORTOLETO,Audi,,F50537,BOR
6,Isack HADJAR,Red Bull Racing,,4781D7,HAD
10,Pierre GASLY,Alpine,,00A1E8,GAS
11,Sergio PEREZ,Cadillac,,909090,PER
12,Kimi ANTONELLI,Mercedes,,00D7B6,ANT
14,Fernando ALONSO,Aston Martin,,229971,ALO
16,Charles LECLERC,Ferrari,,ED1131,LEC
18,Lance STROLL,Aston Martin,,229971,STR


In [0]:
# 1. Team-wise Driver Count (Insight Dataset)
from pyspark.sql.functions import count

team_analysis_df = presentation_df.groupBy("team_name") \
    .agg(count("*").alias("driver_count"))

display(team_analysis_df)


team_name,driver_count
Williams,2
Audi,2
Cadillac,2
Mercedes,2
McLaren,2
Haas F1 Team,2
Ferrari,2
Red Bull Racing,2
Alpine,2
Aston Martin,2


In [0]:
#  Save to presentation layer
team_analysis_df.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis"
)

In [0]:
# verify presentation layer
verify_df = spark.read \
.format("delta") \
.load(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis"
)

display(verify_df)

team_name,driver_count
Williams,2
Audi,2
Cadillac,2
Mercedes,2
McLaren,2
Haas F1 Team,2
Ferrari,2
Red Bull Racing,2
Alpine,2
Aston Martin,2


In [0]:
# Export as csv
verify_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv"
)

In [0]:
# verify csv
dbutils.fs.ls(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv"
)

[FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv/_committed_1256830666962882437', name='_committed_1256830666962882437', size=202, modificationTime=1782204820000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv/_committed_1387002210244714724', name='_committed_1387002210244714724', size=203, modificationTime=1782204649000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv/_committed_1847562064175417360', name='_committed_1847562064175417360', size=201, modificationTime=1782122625000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv/_committed_1936956730068583816', name='_committed_1936956730068583816', size=201, modificationTime=1782124175000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_analysis_csv/_committed_2370782896555968103', name='_committed_237078289655596

In [0]:
# 2  Team Colour vs Driver Count
from pyspark.sql.functions import countDistinct

team_colour_analysis_df = presentation_df.groupBy(
    "team_name",
    "team_colour").agg(
    countDistinct("driver_number").alias("driver_count")
)

display(team_colour_analysis_df)

team_name,team_colour,driver_count
Haas F1 Team,9C9FA2,2
Mercedes,00D7B6,2
Racing Bulls,6C98FF,2
Ferrari,ED1131,2
Cadillac,909090,2
Audi,F50537,2
Alpine,00A1E8,2
Williams,1868DB,2
Red Bull Racing,4781D7,2
McLaren,F47600,2


In [0]:
# save to presentaton layer
team_colour_analysis_df.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis"
)

In [0]:
# verify presention layer
display(
    spark.read.format("delta").load(
        "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis"
    )
)

team_name,team_colour,driver_count
Haas F1 Team,9C9FA2,2
Mercedes,00D7B6,2
Racing Bulls,6C98FF,2
Ferrari,ED1131,2
Cadillac,909090,2
Audi,F50537,2
Alpine,00A1E8,2
Williams,1868DB,2
Red Bull Racing,4781D7,2
McLaren,F47600,2


In [0]:
# export as csv
team_colour_analysis_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header","true") \
.csv(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv"
)

In [0]:
dbutils.fs.ls(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv"
)

[FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv/_committed_1275859550777215439', name='_committed_1275859550777215439', size=201, modificationTime=1782121893000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv/_committed_2442554189906867256', name='_committed_2442554189906867256', size=201, modificationTime=1782200909000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv/_committed_2864000809465614706', name='_committed_2864000809465614706', size=201, modificationTime=1782199239000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv/_committed_3294345102598788789', name='_committed_3294345102598788789', size=203, modificationTime=1782200940000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/team_colour_analysis_csv/_committed_3356870075729680191

In [0]:
3 # Driver Profile Dataset
driver_profile_df = presentation_df.select(
    "driver_number",
    "full_name",
    "team_name",
    "name_acronym"
).distinct()

display(driver_profile_df)

driver_number,full_name,team_name,name_acronym
41,Arvid LINDBLAD,Racing Bulls,LIN
43,Franco COLAPINTO,Alpine,COL
14,Fernando ALONSO,Aston Martin,ALO
1,Lando NORRIS,McLaren,NOR
31,Esteban OCON,Haas F1 Team,OCO
87,Oliver BEARMAN,Haas F1 Team,BEA
81,Oscar PIASTRI,McLaren,PIA
6,Isack HADJAR,Red Bull Racing,HAD
63,George RUSSELL,Mercedes,RUS
12,Kimi ANTONELLI,Mercedes,ANT


In [0]:
# save to presentation layer
driver_profile_df.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile"
)

In [0]:
# verfy presentation layer
verify_df = spark.read \
.format("delta") \
.load(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile"
)

display(verify_df)

driver_number,full_name,team_name,name_acronym
41,Arvid LINDBLAD,Racing Bulls,LIN
43,Franco COLAPINTO,Alpine,COL
14,Fernando ALONSO,Aston Martin,ALO
1,Lando NORRIS,McLaren,NOR
31,Esteban OCON,Haas F1 Team,OCO
87,Oliver BEARMAN,Haas F1 Team,BEA
81,Oscar PIASTRI,McLaren,PIA
6,Isack HADJAR,Red Bull Racing,HAD
63,George RUSSELL,Mercedes,RUS
12,Kimi ANTONELLI,Mercedes,ANT


In [0]:
# export as csv
driver_profile_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv"
)

In [0]:
# check csv folder
dbutils.fs.ls(
    "abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv"
)

[FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv/_committed_1009309563365205165', name='_committed_1009309563365205165', size=200, modificationTime=1782121518000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv/_committed_1094247358088353323', name='_committed_1094247358088353323', size=201, modificationTime=1782121711000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv/_committed_1187419612601065568', name='_committed_1187419612601065568', size=203, modificationTime=1782199398000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv/_committed_1623814242721457809', name='_committed_1623814242721457809', size=202, modificationTime=1782199245000),
 FileInfo(path='abfss://presentation@driversdatastorage.dfs.core.windows.net/driver_profile_csv/_committed_20539461602180565', name='_committed_205394616021